# Question Data from Metaculus API (v2 - Rate Limit Safe)

**Date:** 2026-02-11  
**Version:** 010a - Fixed rate limiting + retry logic + progress saves  
**Input:** `products/Run_Question_Map_2026-02-10_v01.csv`  
**Output:** `products/Question_Data_from_API_2026-02-11_vNN.csv`

**Improvements over 010:**
- ✅ Slower rate limit (2s between requests)
- ✅ Exponential backoff retry on 429 errors
- ✅ Periodic progress saves every 25 questions
- ✅ Resume capability (skips already-fetched questions)
- ✅ Versioned output files (no overwrites)

In [1]:
# Imports
import requests
import pandas as pd
import time
import json
from pathlib import Path
from datetime import date
from typing import Dict, Optional

print("✅ Imports successful")

✅ Imports successful


In [2]:
# Configuration
INPUT_FILE = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products/Run_Question_Map_2026-02-10_v01.csv")
OUTPUT_DIR = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products")
API_BASE = "https://www.metaculus.com/api2/questions"
TEST_LIMIT = None  # Set to None for all questions

# Rate limiting settings
RATE_LIMIT_DELAY = 2.0  # Seconds between requests (increased from 0.5)
MAX_RETRIES = 3  # Max retry attempts on 429 error
BACKOFF_BASE = 5  # Base delay for exponential backoff (seconds)
PROGRESS_SAVE_INTERVAL = 25  # Save progress every N questions

print(f"Input file: {INPUT_FILE}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Test limit: {TEST_LIMIT}")
print(f"Rate limit: {RATE_LIMIT_DELAY}s between requests")
print(f"Max retries: {MAX_RETRIES}")
print(f"Progress saves: every {PROGRESS_SAVE_INTERVAL} questions")

Input file: C:\Users\Donni\projects\metac_bot_Spring_2026\products\Run_Question_Map_2026-02-10_v01.csv
Output dir: C:\Users\Donni\projects\metac_bot_Spring_2026\products
Test limit: None
Rate limit: 2.0s between requests
Max retries: 3
Progress saves: every 25 questions


In [3]:
# Load question list
df_runs = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df_runs)} run records")

# Extract unique question numbers (filter out empty values)
question_numbers = df_runs['question_number'].dropna().astype(int).unique().tolist()
question_numbers.sort()

print(f"Found {len(question_numbers)} unique questions")
print(f"Range: {min(question_numbers)} to {max(question_numbers)}")
print(f"First 10: {question_numbers[:10]}")

Loaded 1260 run records
Found 193 unique questions
Range: 41379 to 42078
First 10: [41379, 41380, 41382, 41383, 41384, 41385, 41386, 41389, 41392, 41396]


In [4]:
# API fetch function with retry logic
def fetch_question_data(question_id: int, max_retries: int = MAX_RETRIES) -> Optional[Dict]:
    """Fetch question data from Metaculus API with exponential backoff retry."""
    url = f"{API_BASE}/{question_id}/"
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=30)
            
            # Success
            if response.status_code == 200:
                return response.json()
            
            # Rate limit - retry with exponential backoff
            if response.status_code == 429:
                wait_time = BACKOFF_BASE * (2 ** attempt)
                print(f"\n  ⏳ Rate limited, waiting {wait_time}s... ", end='')
                time.sleep(wait_time)
                continue
            
            # Other error
            response.raise_for_status()
            
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                print(f"\n  ⚠️  API error after {max_retries} attempts: {e}")
                return None
            # Wait before retry
            time.sleep(2)
    
    return None

print("✅ fetch_question_data() with retry logic defined")

✅ fetch_question_data() with retry logic defined


In [5]:
# Data extraction function (same as 010)
def extract_question_fields(data: Dict) -> Dict:
    """Extract relevant fields from API response."""
    
    # Top-level fields
    result = {
        'question_id': data.get('id'),
        'title': data.get('title', ''),
        'short_title': data.get('short_title', ''),
        'slug': data.get('slug', ''),
        'status': data.get('status', ''),
        'resolved': data.get('resolved', False),
        'comment_count': data.get('comment_count', 0),
        'nr_forecasters': data.get('nr_forecasters', 0),
        'forecasts_count': data.get('forecasts_count', 0),
        'author_username': data.get('author_username', ''),
        'curation_status': data.get('curation_status', ''),
    }
    
    # Dates
    result['created_at'] = data.get('created_at', '')
    result['published_at'] = data.get('published_at', '')
    result['edited_at'] = data.get('edited_at', '')
    result['open_time'] = data.get('open_time', '')
    result['actual_close_time'] = data.get('actual_close_time', '')
    result['scheduled_close_time'] = data.get('scheduled_close_time', '')
    result['actual_resolve_time'] = data.get('actual_resolve_time', '')
    result['scheduled_resolve_time'] = data.get('scheduled_resolve_time', '')
    
    # Question sub-object
    question = data.get('question', {})
    result['question_type'] = question.get('type', '')
    result['resolution'] = question.get('resolution')
    result['resolution_set_time'] = question.get('resolution_set_time', '')
    result['question_weight'] = question.get('question_weight', '')
    result['description'] = question.get('description', '')
    result['resolution_criteria'] = question.get('resolution_criteria', '')
    result['fine_print'] = question.get('fine_print', '')
    
    # Question-type specific fields
    if result['question_type'] == 'multiple_choice':
        result['mc_options'] = json.dumps(question.get('options', []))
    else:
        result['mc_options'] = ''
    
    if result['question_type'] == 'numeric':
        scaling = question.get('scaling', {})
        result['numeric_range_min'] = scaling.get('range_min', '')
        result['numeric_range_max'] = scaling.get('range_max', '')
        result['open_upper_bound'] = scaling.get('open_upper_bound', '')
        result['open_lower_bound'] = scaling.get('open_lower_bound', '')
    else:
        result['numeric_range_min'] = ''
        result['numeric_range_max'] = ''
        result['open_upper_bound'] = ''
        result['open_lower_bound'] = ''
    
    # Tournament info
    projects = data.get('projects', {})
    default_project = projects.get('default_project', {})
    result['tournament_id'] = default_project.get('id', '')
    result['tournament_name'] = default_project.get('name', '')
    result['tournament_slug'] = default_project.get('slug', '')
    
    # Community forecast from aggregations
    aggregations = data.get('aggregations', {})
    unweighted = aggregations.get('unweighted', {})
    latest = unweighted.get('latest', {})
    
    result['community_forecaster_count'] = latest.get('forecaster_count', '')
    forecast_values = latest.get('forecast_values', [])
    
    # Format community forecast based on type
    if result['question_type'] == 'binary' and len(forecast_values) == 2:
        result['community_forecast'] = f"{forecast_values[1]:.1%}"  # p_yes
        result['community_forecast_mean'] = forecast_values[1]
    elif result['question_type'] == 'multiple_choice':
        result['community_forecast'] = json.dumps(forecast_values)
        result['community_forecast_mean'] = ''
    elif result['question_type'] == 'numeric':
        means = latest.get('means', [])
        if means:
            result['community_forecast_mean'] = means[0]
            result['community_forecast'] = f"{means[0]:.2f}"
        else:
            result['community_forecast'] = ''
            result['community_forecast_mean'] = ''
    else:
        result['community_forecast'] = ''
        result['community_forecast_mean'] = ''
    
    # Interval bounds
    interval_lower = latest.get('interval_lower_bounds', [])
    interval_upper = latest.get('interval_upper_bounds', [])
    result['community_interval_lower'] = interval_lower[0] if interval_lower else ''
    result['community_interval_upper'] = interval_upper[0] if interval_upper else ''
    
    # Score data
    score_data = latest.get('score_data', {})
    result['coverage'] = score_data.get('coverage', '')
    result['peer_score'] = score_data.get('peer_score', '')
    result['baseline_score'] = score_data.get('baseline_score', '')
    result['spot_peer_score'] = score_data.get('spot_peer_score', '')
    result['spot_baseline_score'] = score_data.get('spot_baseline_score', '')
    
    return result

print("✅ extract_question_fields() defined")

✅ extract_question_fields() defined


In [6]:
# Helper: Find next version number for output file
def get_next_version_file(base_name: str) -> Path:
    """Get next available version number for output file."""
    version = 1
    while True:
        filename = OUTPUT_DIR / f"{base_name}_v{version:02d}.csv"
        if not filename.exists():
            return filename
        version += 1

# Helper: Check if question already fetched (for resume capability)
def load_existing_results(base_name: str) -> pd.DataFrame:
    """Load most recent output file if it exists."""
    import glob
    pattern = str(OUTPUT_DIR / f"{base_name}_v*.csv")
    files = sorted(glob.glob(pattern))
    if files:
        print(f"Found existing file: {Path(files[-1]).name}")
        return pd.read_csv(files[-1])
    return pd.DataFrame()

print("✅ Helper functions defined")

✅ Helper functions defined


In [7]:
# Test on single question
test_id = question_numbers[0]
print(f"Testing API fetch for question {test_id}...")

test_data = fetch_question_data(test_id)
if test_data:
    print(f"✅ API response received ({len(test_data)} top-level keys)")
    test_fields = extract_question_fields(test_data)
    print(f"✅ Extracted {len(test_fields)} fields")
    print("\nSample fields:")
    for k in ['question_id', 'title', 'question_type', 'status', 'resolved', 'resolution']:
        print(f"  {k}: {test_fields.get(k)}")
else:
    print("❌ API fetch failed")

Testing API fetch for question 41379...
✅ API response received (31 top-level keys)
✅ Extracted 44 fields

Sample fields:
  question_id: 41379
  title: Will the interest in “el salvador” change between 2026-01-05 and 2026-01-17 according to Google Trends?
  question_type: multiple_choice
  status: resolved
  resolved: True
  resolution: Doesn't change


In [8]:
# Batch fetch with improved rate limiting and progress saves
questions_to_fetch = question_numbers[:TEST_LIMIT] if TEST_LIMIT else question_numbers

# Check for existing results (resume capability)
base_name = f"Question_Data_from_API_{date.today()}"
existing_df = load_existing_results(base_name)
already_fetched = set(existing_df['question_id'].tolist()) if not existing_df.empty else set()

if already_fetched:
    print(f"\nResuming: {len(already_fetched)} questions already fetched")
    questions_to_fetch = [q for q in questions_to_fetch if q not in already_fetched]
    print(f"Remaining: {len(questions_to_fetch)} questions\n")
else:
    print(f"\nFetching {len(questions_to_fetch)} questions...\n")

results = existing_df.to_dict('records') if not existing_df.empty else []
failed = []
last_save_count = len(results)

for i, qnum in enumerate(questions_to_fetch, 1):
    print(f"[{i}/{len(questions_to_fetch)}] Q{qnum}...", end='')
    
    data = fetch_question_data(qnum)
    if data:
        try:
            fields = extract_question_fields(data)
            results.append(fields)
            print(f" ✓ ({fields['question_type']}, {fields['status']})")
        except Exception as e:
            print(f" ⚠️  Extract failed: {e}")
            failed.append(qnum)
    else:
        print(f" ❌ Fetch failed")
        failed.append(qnum)
    
    # Periodic progress save
    if (len(results) - last_save_count) >= PROGRESS_SAVE_INTERVAL:
        temp_df = pd.DataFrame(results)
        progress_file = OUTPUT_DIR / f"{base_name}_progress.csv"
        temp_df.to_csv(progress_file, index=False)
        print(f"  💾 Progress saved: {len(results)} questions")
        last_save_count = len(results)
    
    # Rate limiting (be respectful)
    if i < len(questions_to_fetch):
        time.sleep(RATE_LIMIT_DELAY)

print(f"\n✅ Successfully fetched {len(results)} total questions")
if failed:
    print(f"❌ Failed to fetch {len(failed)} questions: {failed}")


Fetching 193 questions...

 ✓ (multiple_choice, resolved)
[2/193] Q41380... ✓ (numeric, resolved)
[3/193] Q41382... ✓ (multiple_choice, resolved)
[4/193] Q41383... ✓ (numeric, resolved)
[5/193] Q41384... ✓ (multiple_choice, resolved)
[6/193] Q41385... ✓ (numeric, resolved)
[7/193] Q41386... ✓ (multiple_choice, resolved)
[8/193] Q41389... ✓ (multiple_choice, resolved)
[9/193] Q41392... ✓ (numeric, resolved)
[10/193] Q41396... ✓ (binary, resolved)
[11/193] Q41400... ✓ (binary, resolved)
[12/193] Q41403... ✓ (multiple_choice, resolved)
[13/193] Q41404... ✓ (binary, resolved)
[14/193] Q41406... ✓ (numeric, resolved)
[15/193] Q41408... ✓ (numeric, resolved)
[16/193] Q41409... ✓ (binary, resolved)
[17/193] Q41410... ✓ (numeric, resolved)
[18/193] Q41411... ✓ (binary, resolved)
[19/193] Q41412... ✓ (multiple_choice, resolved)
[20/193] Q41413... ✓ (binary, resolved)
[21/193] Q41414... ✓ (binary, resolved)
[22/193] Q41415... ✓ (binary, resolved)
[23/193] Q41416... ✓ (binary, resolved)
[24/193]

In [9]:
# Convert to DataFrame
df = pd.DataFrame(results)
print(f"DataFrame shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}): {list(df.columns)}")

DataFrame shape: (192, 44)

Columns (44): ['question_id', 'title', 'short_title', 'slug', 'status', 'resolved', 'comment_count', 'nr_forecasters', 'forecasts_count', 'author_username', 'curation_status', 'created_at', 'published_at', 'edited_at', 'open_time', 'actual_close_time', 'scheduled_close_time', 'actual_resolve_time', 'scheduled_resolve_time', 'question_type', 'resolution', 'resolution_set_time', 'question_weight', 'description', 'resolution_criteria', 'fine_print', 'mc_options', 'numeric_range_min', 'numeric_range_max', 'open_upper_bound', 'open_lower_bound', 'tournament_id', 'tournament_name', 'tournament_slug', 'community_forecaster_count', 'community_forecast', 'community_forecast_mean', 'community_interval_lower', 'community_interval_upper', 'coverage', 'peer_score', 'baseline_score', 'spot_peer_score', 'spot_baseline_score']


In [10]:
# Data quality checks
print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

print(f"\nQuestion Types:")
print(df['question_type'].value_counts())

print(f"\nStatus:")
print(df['status'].value_counts())

print(f"\nResolved:")
print(df['resolved'].value_counts())

print(f"\nResolution completeness (for resolved questions):")
resolved_df = df[df['resolved'] == True]
print(f"  Total resolved: {len(resolved_df)}")
print(f"  With resolution value: {resolved_df['resolution'].notna().sum()}")
print(f"  Missing resolution: {resolved_df['resolution'].isna().sum()}")

print(f"\nForecaster counts:")
print(f"  With nr_forecasters > 0: {(df['nr_forecasters'] > 0).sum()}")
print(f"  With community_forecaster_count > 0: {(df['community_forecaster_count'] != '').sum()}")

print(f"\nComment counts:")
print(f"  With comment_count > 0: {(df['comment_count'] > 0).sum()}")

print(f"\nCoverage/Score data:")
print(f"  With coverage: {(df['coverage'] != '').sum()}")
print(f"  With peer_score: {(df['peer_score'] != '').sum()}")

print(f"\nTournaments:")
print(df['tournament_name'].value_counts())

DATA QUALITY CHECKS

Question Types:
question_type
binary             91
numeric            59
multiple_choice    36
discrete            6
Name: count, dtype: int64

Status:
status
resolved    101
closed       91
Name: count, dtype: int64

Resolved:
resolved
True     101
False     91
Name: count, dtype: int64

Resolution completeness (for resolved questions):
  Total resolved: 101
  With resolution value: 101
  Missing resolution: 0

Forecaster counts:
  With nr_forecasters > 0: 191
  With community_forecaster_count > 0: 0

Comment counts:
  With comment_count > 0: 105

Coverage/Score data:
  With coverage: 0
  With peer_score: 0

Tournaments:
tournament_name
Spring 2026 AI Forecasting Benchmark Tournament    73
MiniBench - 2026-01-19                             47
MiniBench - 2026-01-05                             44
MiniBench                                          28
Name: count, dtype: int64


In [11]:
# Save to VERSIONED CSV
output_file = get_next_version_file(base_name)
df.to_csv(output_file, index=False)
print(f"\n✅ Saved to: {output_file.name}")
print(f"   Rows: {len(df)}")
print(f"   Columns: {len(df.columns)}")
print(f"   Size: {output_file.stat().st_size / 1024:.1f} KB")

# Clean up progress file if it exists
progress_file = OUTPUT_DIR / f"{base_name}_progress.csv"
if progress_file.exists():
    progress_file.unlink()
    print(f"   Cleaned up progress file")


✅ Saved to: Question_Data_from_API_2026-02-11_v01.csv
   Rows: 192
   Columns: 44
   Size: 108.4 KB
   Cleaned up progress file


In [12]:
# Display sample of key fields
print("\n" + "=" * 80)
print("SAMPLE DATA (first 10 questions)")
print("=" * 80)

key_cols = ['question_id', 'short_title', 'question_type', 'status', 'resolved', 
            'resolution', 'nr_forecasters', 'community_forecast']
available_cols = [c for c in key_cols if c in df.columns]
df[available_cols].head(10)


SAMPLE DATA (first 10 questions)


,question_id,short_title,question_type,status,resolved,resolution,nr_forecasters,community_forecast
0,41379,"Will Google Trend topic ""el salvador"" rise?",multiple_choice,resolved,True,Doesn't change,76,[]
1,41380,What will the value of FRED series SOFRINDEX be?,numeric,resolved,True,1.22822,75,
2,41382,"Will Google Trend topic ""newburgh"" rise?",multiple_choice,resolved,True,Decreases,79,[]
3,41383,What will the value of FRED series BAMLC0A0CME...,numeric,resolved,True,4.84,82,
4,41384,"Will Google Trend topic ""taiwan"" rise?",multiple_choice,resolved,True,Decreases,78,[]
5,41385,What will the value of FRED series DBAA be?,numeric,resolved,True,5.87,82,
6,41386,"Will Google Trend topic ""somali daycare minnes...",multiple_choice,resolved,True,Decreases,78,[]
7,41389,"Will Google Trend topic ""maría corina machado""...",multiple_choice,resolved,True,Decreases,77,[]
8,41392,What will the value of FRED series BAMLH0A1HYB...,numeric,resolved,True,5.41,84,
9,41396,APH's close price rises?,binary,resolved,True,yes,81,


In [13]:
available_cols

['question_id',
 'short_title',
 'question_type',
 'status',
 'resolved',
 'resolution',
 'nr_forecasters',
 'community_forecast']